# Member 1 — Exact CVRP Method | Version 3: Experiments and Results
**Git commit:** `feat(exact): add reproducible solver experiments and result export`

Runs the exact method on a few problem sizes using the same data-generation convention, records runtime/status/objective/bound, validates routes, and exports a CSV. Exact runtimes vary by Colab hardware; do not copy illustrative values into the report.

## 1. Dependencies and shared data

In [ ]:
!pip -q install numpy pandas matplotlib ortools

In [ ]:
import math, random, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
N_CUSTOMERS = 20   # Increase to 31 to match the scale of the sample report
K = 5
CAPACITY = 100
rng = np.random.default_rng(SEED)

# Shared deterministic synthetic instance. Both members must use these same parameters.
coords = [(50.0, 50.0)] + [(float(x), float(y)) for x,y in rng.uniform(0,100,size=(N_CUSTOMERS,2))]
demands = [0] + [int(x) for x in rng.integers(4, 11, size=N_CUSTOMERS)]
n = len(coords)
dist = np.zeros((n,n), dtype=int)
for i in range(n):
    for j in range(n):
        if i != j:
            dist[i,j] = int(math.floor(math.dist(coords[i], coords[j]) + 0.5))

print(f"Nodes: {n} (including depot 0); customers: {N_CUSTOMERS}; vehicles: {K}; capacity: {CAPACITY}")
print(f"Total demand: {sum(demands)}; total fleet capacity: {K*CAPACITY}")
assert sum(demands) <= K*CAPACITY, "Infeasible generated instance; adjust seed/demands."

In [ ]:
def route_distance(route):
    """Distance for a customer list; depot 0 is added at both ends."""
    seq = [0] + list(route) + [0]
    return int(sum(dist[seq[i], seq[i+1]] for i in range(len(seq)-1)))

def solution_cost(routes):
    return sum(route_distance(r) for r in routes)

def validate_routes(routes, verbose=True):
    errors = []
    seen = [c for route in routes for c in route]
    if sorted(seen) != list(range(1,n)):
        errors.append("Every customer must appear exactly once.")
    if len(routes) > K:
        errors.append(f"Used {len(routes)} routes, exceeding fleet limit {K}.")
    for ridx, route in enumerate(routes, start=1):
        load = sum(demands[c] for c in route)
        if load > CAPACITY:
            errors.append(f"Route {ridx} load {load} exceeds capacity {CAPACITY}.")
        if not route:
            errors.append(f"Route {ridx} is empty.")
    cost = solution_cost(routes)
    if verbose:
        print("Feasible:", not errors, "| Cost:", cost, "| Routes:", len(routes))
        for e in errors: print(" -", e)
    return {"feasible": not errors, "cost": cost, "routes_used": len(routes), "errors": errors}

## 2. CP-SAT solver

In [ ]:
from ortools.sat.python import cp_model

def solve_cvrp_cpsat(time_limit_s=45, workers=8):
    model = cp_model.CpModel()
    x, arcs = {}, []
    for i in range(n):
        for j in range(n):
            if i != j:
                x[i,j] = model.NewBoolVar(f"x_{i}_{j}")
                arcs.append((i,j,x[i,j]))
    model.AddMultipleCircuit(arcs)
    load = {i:model.NewIntVar(demands[i], CAPACITY, f"load_{i}") for i in range(1,n)}
    for i in range(1,n):
        for j in range(1,n):
            if i != j:
                model.Add(load[j] >= load[i] + demands[j]).OnlyEnforceIf(x[i,j])
    model.Add(sum(x[0,j] for j in range(1,n)) <= K)
    model.Minimize(sum(int(dist[i,j])*x[i,j] for (i,j) in x))
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = float(time_limit_s)
    solver.parameters.num_search_workers = int(workers)
    solver.parameters.random_seed = SEED
    t0=time.perf_counter(); status=solver.Solve(model); elapsed=time.perf_counter()-t0
    out={"method":"CP-SAT","status":solver.StatusName(status),"runtime_s":elapsed,
         "objective":None,"best_bound":None,"optimality_proven":status==cp_model.OPTIMAL,
         "feasible":False,"routes":[]}
    if status in (cp_model.OPTIMAL,cp_model.FEASIBLE):
        succ={}
        for (i,j),v in x.items():
            if solver.Value(v): succ.setdefault(i,[]).append(j)
        routes=[]; remaining=set(range(1,n))
        for first in succ.get(0,[]):
            route=[]; cur=first
            while cur != 0 and cur not in route:
                route.append(cur); remaining.discard(cur)
                cur=succ[cur][0]
            if route: routes.append(route)
        val=validate_routes(routes,verbose=False)
        out.update(objective=int(round(solver.ObjectiveValue())),
                   best_bound=float(solver.BestObjectiveBound()),
                   feasible=val["feasible"],routes=routes)
        if remaining: print("Extraction warning:", sorted(remaining))
    return out

exact_result=solve_cvrp_cpsat()
print({k:v for k,v in exact_result.items() if k!="routes"})

## 3. Optional size experiment

In [ ]:
# Use small-to-moderate sizes for a quick Colab run. Increase to 31 only if runtime permits.
rows=[]
for customers in [10, 15, 20]:
    N_CUSTOMERS = customers
    rng = np.random.default_rng(SEED)
    coords = [(50.0,50.0)] + [(float(x),float(y)) for x,y in rng.uniform(0,100,size=(N_CUSTOMERS,2))]
    demands = [0] + [int(x) for x in rng.integers(4,11,size=N_CUSTOMERS)]
    n=len(coords)
    dist=np.zeros((n,n),dtype=int)
    for i in range(n):
        for j in range(n):
            if i!=j: dist[i,j]=int(math.floor(math.dist(coords[i],coords[j])+0.5))
    # Use a capacity/fleet limit that is feasible for the generated instance.
    K=max(3, math.ceil(sum(demands[1:])/45))
    CAPACITY=max(45, math.ceil(sum(demands[1:])/K))
    # Rebuild solver locally by using current global n, dist, demands, K, CAPACITY.
    res=solve_cvrp_cpsat(time_limit_s=30)
    rows.append({"customers":customers,"status":res["status"],"objective":res["objective"],
                 "best_bound":res["best_bound"],"runtime_s":round(res["runtime_s"],3),
                 "optimality_proven":res["optimality_proven"],"feasible":res["feasible"]})
exact_experiments=pd.DataFrame(rows)
display(exact_experiments)
exact_experiments.to_csv("member1_exact_results.csv",index=False)

## 4. Commit and handoff
- Commit this notebook and the CSV only if the CSV is a deliberately tracked reproducibility artifact.
- Send Member 2 the exact instance-generation parameters and distance convention.
- Final comparison must use the same instance and must independently validate all methods.